In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Open the real AI Assistant page (pinned sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'AI Assistant')]"))).click()
    chat_input = wait.until(EC.visibility_of_element_located((By.ID, "ai-chat-input")))
    print("AI query interface found.")

    # Real input + submit button (verified in ChatInput.jsx)
    chat_input.clear()
    chat_input.send_keys("What medicines are currently low in stock?")
    driver.find_element(By.XPATH, "//button[@type='submit' and contains(., 'Send')]").click()

    # Wait for the answer: typing indicator gone AND an AI Assistant bubble rendered
    long_wait = WebDriverWait(driver, 60)
    long_wait.until(lambda d: not d.find_elements(By.XPATH, "//*[@aria-label='Assistant is typing']")
                    and len(d.find_elements(By.XPATH, "//*[text()='AI Assistant']")) >= 2)
    time.sleep(1)
    body = driver.find_element(By.TAG_NAME, "body").text
    assert "Request failed" not in body, "AI request failed (error bubble shown)."
    assert "What medicines are currently low in stock?" in body, "Query echo missing."
    print("Response displayed (excerpt):", body[body.find("What medicines are currently low in stock?"):][:400])
    print("PASS: AI query answered")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("39_ai_query_FAIL.png")
finally:
    driver.quit()